<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajeevraibhatia/agent-harness-evals/blob/main/notebooks/m8_capstone.ipynb)
[![Course](https://img.shields.io/badge/Full%20Course-rajeevraibhatia.com-7c3aed)](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-8)

</div>

# Module 8 — Capstone: Document Research Agent

**Level:** Advanced | **Time:** ~3h | [Full reading →](https://rajeevraibhatia.com/curriculum/agent-harness-evals#module-8)

### What you'll build
End-to-end Document Research Agent combining all previous modules:
- **ToolRegistry** (M2): schema validation, idempotency gates
- **MemoryManager** (M3): episodic store of past research sessions
- **Harness** (M5): initializer/executor, replay log, circuit breaker
- **EvalSuite** (M6): 5-task suite, code + LLM graders, pass@k scoring

### Agent spec
- **Input**: research question (string)
- **Tools**: `search_web`, `extract_url`, `store_finding`, `generate_answer`
- **Output**: `{answer: str, citations: list, confidence: float}`
- **Eval target**: pass@3 ≥ 0.85

### Stretch goals
1. Add a critic agent that reviews the draft answer before final output
2. Add confidence calibration: compare stated confidence to actual accuracy
3. Run your eval harness on [GAIA Level 1](https://huggingface.co/datasets/gaia-benchmark/GAIA) tasks

---

In [ ]:
!pip install openai --quiet

In [ ]:
"""
Capstone: Document Research Agent

Combines ToolRegistry (M2) + MemoryManager (M3) + Harness (M5) + EvalSuite (M6)
into a complete working agent.

Input:  research question (string)
Output: {answer: str, citations: list, confidence: float}
"""
import os, json, time, math, re, hashlib
from dataclasses import dataclass, field
from typing import Optional, Callable
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ── Simplified ToolRegistry (from M2) ─────────────────────────────────────────

class ToolRegistry:
    def __init__(self):
        self._tools = {}

    def register(self, fn, schema, idempotent=True):
        self._tools[schema["name"]] = {"fn": fn, "schema": schema, "idempotent": idempotent}

    @property
    def schemas(self):
        return [{"type": "function", "function": t["schema"]} for t in self._tools.values()]

    def execute(self, name, args):
        if name not in self._tools:
            return f"ERROR: tool '{name}' not found"
        try:
            return str(self._tools[name]["fn"](**args))
        except Exception as e:
            return f"ERROR: {name} failed — {e}"

# ── Tool implementations ──────────────────────────────────────────────────────

_findings_store = []

def search_web(query: str, num_results: int = 3) -> str:
    """Mock web search — replace with real search API."""
    results = {
        "transformer architecture": "Transformers use self-attention. Vaswani et al. (2017) introduced the architecture in 'Attention Is All You Need'.",
        "bert fine-tuning": "BERT fine-tuning uses AdamW optimizer, lr=2e-5, 3 epochs. Works well for classification and NER.",
        "gpt tokenizer": "GPT uses Byte-Pair Encoding (BPE). GPT-4 uses cl100k_base with ~100k tokens.",
        "attention mechanism": "Attention = softmax(QK^T / sqrt(d_k)) * V. Multi-head attention runs h parallel attention heads.",
        "rag retrieval": "RAG combines dense retrieval (FAISS, Pinecone) with a generative LLM. Chunk size 512, overlap 64 is common.",
    }
    query_lower = query.lower()
    for key, result in results.items():
        if any(w in query_lower for w in key.split()):
            return result
    return f"No results found for: {query}"

def extract_url(url: str, max_chars: int = 1000) -> str:
    return f"[Extracted content from {url}]: Sample content about the topic. (Mock — replace with real HTTP fetch + parse)"

def store_finding(source: str, passage: str, relevance_score: float) -> str:
    _findings_store.append({"source": source, "passage": passage, "relevance": relevance_score})
    return f"Stored finding #{len(_findings_store)}"

def generate_answer(question: str) -> dict:
    if not _findings_store:
        return {"answer": "No findings available.", "citations": [], "confidence": 0.0}
    findings_text = "\n".join(f"[{f['relevance']:.1f}] {f['source']}: {f['passage']}" for f in _findings_store)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a research synthesizer. Given findings, produce a structured answer. Return JSON: {answer: str, citations: [str], confidence: 0-1}"},
            {"role": "user", "content": f"Question: {question}\n\nFindings:\n{findings_text}"}
        ],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)


# ── Document Research Agent ───────────────────────────────────────────────────

class DocumentResearchAgent:
    MAX_STEPS = 15

    def __init__(self):
        self.registry = ToolRegistry()
        self._register_tools()
        _findings_store.clear()

    def _register_tools(self):
        self.registry.register(search_web, {
            "name": "search_web",
            "description": "Search the web. Be specific — 'BERT fine-tuning classification 2024' beats 'BERT training'.",
            "parameters": {"type": "object", "properties": {
                "query": {"type": "string"},
                "num_results": {"type": "integer", "default": 3}
            }, "required": ["query"]}
        }, idempotent=True)

        self.registry.register(store_finding, {
            "name": "store_finding",
            "description": "Store a research finding with source, passage, and relevance score (0-1).",
            "parameters": {"type": "object", "properties": {
                "source": {"type": "string"},
                "passage": {"type": "string"},
                "relevance_score": {"type": "number"}
            }, "required": ["source", "passage", "relevance_score"]}
        }, idempotent=False)

        self.registry.register(generate_answer, {
            "name": "generate_answer",
            "description": "Synthesize stored findings into a final answer. Call only after storing 3+ findings.",
            "parameters": {"type": "object", "properties": {
                "question": {"type": "string"}
            }, "required": ["question"]}
        }, idempotent=True)

    def research(self, question: str) -> dict:
        messages = [
            {"role": "system", "content": (
                "You are a research agent. For the given question:\n"
                "1. Search for 3-5 relevant sources using search_web\n"
                "2. Store the most relevant passages using store_finding\n"
                "3. Call generate_answer to synthesize findings\n"
                "Be thorough. Cite your sources."
            )},
            {"role": "user", "content": question}
        ]

        for step in range(self.MAX_STEPS):
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
                tools=self.registry.schemas,
                tool_choice="auto"
            )
            msg = response.choices[0].message
            messages.append(msg)

            if not msg.tool_calls:
                return {"answer": msg.content, "citations": [], "confidence": 0.5}

            for tc in msg.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments)
                print(f"  [{step}] {name}({list(args.keys())})")
                result = self.registry.execute(name, args)

                if name == "generate_answer" and not result.startswith("ERROR"):
                    try:
                        return json.loads(result)
                    except Exception:
                        pass

                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})

        return {"answer": "Research incomplete — max steps reached.", "citations": [], "confidence": 0.0}


# ── Run demo ──────────────────────────────────────────────────────────────────

agent = DocumentResearchAgent()
question = "How does the transformer attention mechanism work?"
print(f"Question: {question}\n")
result = agent.research(question)
print(f"\nAnswer: {result.get('answer', 'N/A')[:300]}")
print(f"Citations: {result.get('citations', [])}")
print(f"Confidence: {result.get('confidence', 0):.2f}")

## Eval Suite

Run 5 eval tasks with pass@k scoring.

In [ ]:
# Eval suite for the Document Research Agent

def pass_at_k(p, k):
    return 1.0 - (1.0 - p) ** k

EVAL_TASKS = [
    {"id": "t1", "question": "What is Byte-Pair Encoding?", "must_contain": ["bpe", "byte", "encoding", "tokenizer"]},
    {"id": "t2", "question": "How does RAG work?", "must_contain": ["retrieval", "generation", "augmented"]},
    {"id": "t3", "question": "What is the transformer attention formula?", "must_contain": ["softmax", "query", "key", "value"]},
    {"id": "t4", "question": "What optimizer is used for BERT fine-tuning?", "must_contain": ["adamw", "adam"]},
    {"id": "t5", "question": "What is multi-head attention?", "must_contain": ["head", "parallel", "attention"]},
]

def code_grade(answer: str, must_contain: list[str]) -> float:
    ans_lower = answer.lower()
    hits = sum(1 for kw in must_contain if kw in ans_lower)
    return hits / len(must_contain)

# Simulate: run agent 3 times per task (in practice, each trial is independent)
import random
random.seed(42)

print("Simulated eval (3 trials per task):")
print(f"{'Task':<6} {'p_hat':<8} {'pass@3':<10} {'pass^3':<10}")
print("-" * 40)

for task in EVAL_TASKS:
    # Simulate stochastic results with ~0.8 success rate
    trials = [random.random() < 0.8 for _ in range(3)]
    p_hat = sum(trials) / len(trials)
    print(f"{task['id']:<6} {p_hat:<8.2f} {pass_at_k(p_hat, 3):<10.3f} {p_hat**3:<10.3f}")

print("\nStretch goal: connect to your real DocumentResearchAgent and run these tasks.")
print("Target: pass@3 >= 0.85 across all 5 tasks.")

## What's Next?

Congratulations on completing the course! You've built:
- A ReAct loop from scratch
- A production-grade ToolRegistry with safety properties
- A 3-tier MemoryManager
- A multi-agent supervisor + producer-critic loop
- A full harness with replay log and circuit breaker
- An eval suite with pass@k scoring and LLM-as-judge
- A complete Document Research Agent

**Share your work**: Star the repo, share on LinkedIn, apply these patterns at work.

[→ Full course reading](https://rajeevraibhatia.com/curriculum/agent-harness-evals) | [→ All modules](https://github.com/rajeevraibhatia/agent-harness-evals)